# Neuro-Symbolic PPM — Sepsis_Case (T-LEAF · label-noise robustness)

**The question.** Following the PPM pipeline `event log → prefix log → encoding → learning → validation → runtime prediction`, can **clean symbolic temporal knowledge** keep a neural next-activity model accurate and process-conformant when its **training labels are corrupted**?

We mine the symbolic layer — an empirical **directly-follows DFA** and **LTLf precedence** rules `a precedes b ≡ (¬b U a) ∨ G(¬b)` — from the **clean** training traces, then corrupt a growing fraction (10/20/30%) of the **training targets** with a random DFA activity and retrain. Validation and test always stay clean.

Three variants, two architectures (GRU, LSTM):
- **baseline** — cross-entropy only;
- **baseline+mask** — baseline weights with DFA-forbidden classes removed at inference (output refinement: conformance by construction);
- **checker** — trained with the differentiable forbidden-mass logic loss (T-LEAF *checker* branch).

> Background skills: `ppm-skills` (PPM pipeline: prefix log, encoding, validation) and `TLEAF` (LTLf, DFA embedding, logic loss). The *embedder* branch is described in §5 but deferred from this grid.
>
> **For now results are shown as numbers only**; the cross-dataset plots live in the separate comparison notebook.

In [1]:
%matplotlib inline
import json
import sys
from pathlib import Path

import pandas as pd
import torch

# Make `src.nspm` importable from anywhere under the repository.
ROOT = Path.cwd()
while not (ROOT / "src" / "nspm").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nspm.config import ExperimentConfig, ProjectPaths
from src.nspm.data.xes import read_xes
from src.nspm.data.prefixes import (
    ActivityVocabulary,
    extract_traces,
    split_traces,
)
from src.nspm.pipeline.analysis import build_analysis_tables, summarise
from src.nspm.process.automaton import ProcessDFA
from src.nspm.process.ltl_constraints import mine_precedence_constraints
from src.nspm.learning.logic import build_allowed_mask
from src.nspm.pipeline.benchmark import run_corruption_grid

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

device: cuda


## 1. Configuration

The **only** difference between the three dataset notebooks is `DATASET_NAME`.
`NOISE_LEVELS` is the fraction of training targets that get corrupted;
`FORCE_RERUN=False` reuses cached results so the notebook is cheap to re-open.

In [2]:
# --- experiment parameters (only DATASET_NAME differs across the three notebooks) ---
DATASET_NAME = "Sepsis_Case"

MODELS       = ["gru", "lstm"]                # transformer is wired but off for now
VARIANTS     = ["baseline", "baseline_mask", "checker"]
NOISE_LEVELS = [0.0, 0.1, 0.2, 0.3]           # fraction of training targets corrupted
EPOCHS       = 10
SEED         = 42
DROP_UNSEEN  = True                           # drop held-out cases with unseen activities
MAX_CASES    = None                           # cap cases for a quick smoke run
FORCE_RERUN  = False                          # recompute even if cached results exist

In [3]:
config = ExperimentConfig(seed=SEED)
paths = ProjectPaths.from_root(ROOT, DATASET_NAME)
DATA_PATH = paths.dataset
RESULTS_CSV = ROOT / "runs" / "benchmark_corruption.csv"
print("dataset :", DATASET_NAME)
print("log file:", DATA_PATH.name)

dataset : Sepsis_Case
log file: Sepsis_Cases_Event_Log.xes


## 2. The event log

We load the XES log once and compute the exploratory tables. (We pass
`outcome_markers={}` so the generic case table carries no dataset-specific
outcome flags — those are a Sepsis-only preset.)

In [4]:
events = read_xes(DATA_PATH, max_cases=MAX_CASES)
tables = build_analysis_tables(events, outcome_markers={})
summary = summarise(tables, DATA_PATH)
print(json.dumps({k: summary[k] for k in ["cases", "events", "activities", "variants"]}, indent=2))
display(tables.activities.head(10))

C:\Users\Matteo\miniconda3\envs\tleaf\lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(


{
  "cases": 1050,
  "events": 15214,
  "activities": 16,
  "variants": 846
}


,concept:name,count
0,Leucocytes,3383
1,CRP,3262
2,LacticAcid,1466
3,Admission NC,1182
4,ER Triage,1053
5,ER Registration,1050
6,ER Sepsis Triage,1049
7,IV Antibiotics,823
8,IV Liquid,753
9,Release A,671


## 3. From log to a supervised learning problem

The PPM pipeline turns a log into supervised examples. We **split by case id**
(no case's prefixes leak across train/val/test), build the **activity
vocabulary**, and generate one prefix → next-activity example per event.

In [5]:
traces = extract_traces(events)
splits = split_traces(
    traces,
    validation_fraction=config.data.validation_fraction,
    test_fraction=config.data.test_fraction,
    seed=config.seed,
)
vocab = ActivityVocabulary.from_traces(splits.train.values())
n_prefix = sum(len(t) for t in splits.train.values())
print(f"cases  train / val / test : {len(splits.train)} / {len(splits.validation)} / {len(splits.test)}")
print(f"activities (vocabulary)   : {len(vocab.activities)}")
print(f"training prefix examples  : {n_prefix}")

cases  train / val / test : 735 / 157 / 158
activities (vocabulary)   : 16
training prefix examples  : 10740


## 4. The symbolic knowledge: a DFA and LTLf precedence rules — from the **clean** training set

Two symbolic objects, both mined from *clean* training traces only:

- the empirical **directly-follows DFA** (which activity may directly follow
  which) — it powers the **checker** loss and the inference-time decoding mask;
- mined **LTLf precedence** rules `a precedes b ≡ (¬b U a) ∨ G(¬b)`.

This clean knowledge is held fixed while the training labels get dirtier.

In [6]:
automaton = ProcessDFA.from_traces(splits.train.values())
mask = build_allowed_mask(automaton, vocab)
constraints = mine_precedence_constraints(
    splits.train.values(),
    min_support=max(3, len(splits.train) // 10),
)
print(f"DFA: {len(automaton.states)} states, {automaton.transition_count} transitions")
print(f"test transition coverage: {automaton.transition_coverage(splits.test.values()):.3f}")
print(f"mined LTLf precedence constraints: {len(constraints)}")
display(pd.DataFrame(
    [{"earlier": c.earlier, "later": c.later, "support": c.support,
      "confidence": round(c.confidence, 3)} for c in constraints[:10]]
))

DFA: 18 states, 128 transitions
test transition coverage: 0.999
mined LTLf precedence constraints: 28


,earlier,later,support,confidence
0,ER Sepsis Triage,IV Antibiotics,563,1.0
1,ER Triage,Admission NC,551,1.0
2,ER Registration,Admission NC,551,1.0
3,ER Triage,Release A,465,1.0
4,ER Registration,Release A,465,1.0
5,ER Triage,Return ER,205,1.0
6,ER Registration,Return ER,205,1.0
7,CRP,Return ER,205,1.0
8,Admission NC,Return ER,205,1.0
9,ER Triage,Admission IC,81,1.0


## 5. Embedding the knowledge (the T-LEAF embedder) — *deferred*

T-LEAF can embed each constraint DFA and each predicted trace into a shared
vector space with a hierarchical GNN `q = (qe, qm)` trained by a triplet hinge
loss, giving the logic loss `‖q(A) − q(w_pred)‖²` (skill `TLEAF`). It is the most
expensive branch (one embedder per dataset) and is **not** part of this
noise grid yet — see the project TODO. Here the logic signal is the cheaper,
fully differentiable **checker** (forbidden-mass) loss.

## 6. The corruption grid

`run_corruption_grid` does the heavy lifting: the DFA/vocabulary are built once
from the clean training set, then for every `(model, noise level)` it corrupts
that fraction of training targets, retrains, and evaluates the requested
variants on the clean test set — one tidy row per evaluated model. Cached
results are reused unless `FORCE_RERUN=True`.

In [7]:
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
cached = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else None
have_dataset = cached is not None and (cached["dataset"] == DATASET_NAME).any()

if FORCE_RERUN or not have_dataset:
    grid = run_corruption_grid(
        DATA_PATH, dataset_name=DATASET_NAME,
        model_kinds=tuple(MODELS), noise_levels=tuple(NOISE_LEVELS),
        variants=tuple(VARIANTS), epochs=EPOCHS, seed=SEED,
        drop_unseen=DROP_UNSEEN, max_cases=MAX_CASES, verbose=True,
    )
    others = None if cached is None else cached[cached["dataset"] != DATASET_NAME]
    combined = grid if others is None else pd.concat([others, grid], ignore_index=True)
    combined.to_csv(RESULTS_CSV, index=False)
else:
    grid = cached[cached["dataset"] == DATASET_NAME].reset_index(drop=True)
    print(f"loaded cached results for {DATASET_NAME}")

METRICS = ["accuracy", "macro_f1", "macro_precision", "macro_recall",
           "top_k_accuracy", "violation_rate", "forbidden_mass"]
display(grid[["architecture", "variant", "noise_level"] + METRICS]
        .sort_values(["architecture", "variant", "noise_level"]).round(4))

loaded cached results for Sepsis_Case


,architecture,variant,noise_level,accuracy,macro_f1,macro_precision,macro_recall,top_k_accuracy,violation_rate,forbidden_mass
0,gru,baseline,0.0,0.6739,0.4788,0.5156,0.4718,0.9273,0.0000,0.0033
3,gru,baseline,0.1,0.6716,0.4768,0.5303,0.4740,0.9259,0.0000,0.0318
6,gru,baseline,0.2,0.6679,0.4745,0.5189,0.4703,0.9236,0.0000,0.0601
9,gru,baseline,0.3,0.6707,0.4732,0.4904,0.4683,0.9208,0.0005,0.0874
1,gru,baseline_mask,0.0,0.6739,0.4788,0.5156,0.4718,0.9273,0.0000,0.0000
4,gru,baseline_mask,0.1,0.6716,0.4768,0.5303,0.4740,0.9259,0.0000,0.0000
7,gru,baseline_mask,0.2,0.6679,0.4745,0.5189,0.4703,0.9240,0.0000,0.0000
10,gru,baseline_mask,0.3,0.6711,0.4740,0.4905,0.4698,0.9208,0.0000,0.0000
2,gru,checker,0.0,0.6702,0.4792,0.5082,0.4716,0.9291,0.0000,0.0025
5,gru,checker,0.1,0.6684,0.4814,0.5218,0.4740,0.9273,0.0000,0.0224


## 7. Results by noise regime

Pivot tables (rows = architecture × variant, columns = noise level). Read
**accuracy** (does it survive the noise?) against **forbidden_mass** (does the
clean DFA keep the model conformant?). `baseline+mask` is conformant by
construction (`forbidden_mass = 0`).

In [8]:
for metric in ["accuracy", "macro_f1", "forbidden_mass", "violation_rate"]:
    print(f"\n### {metric}  (rows: architecture x variant | cols: noise level)")
    display(grid.pivot_table(index=["architecture", "variant"],
                             columns="noise_level", values=metric).round(4))


### accuracy  (rows: architecture x variant | cols: noise level)


noise_level                    0.0     0.1     0.2     0.3
architecture variant                                      
gru          baseline       0.6739  0.6716  0.6679  0.6707
             baseline_mask  0.6739  0.6716  0.6679  0.6711
             checker        0.6702  0.6684  0.6721  0.6670
lstm         baseline       0.6735  0.6684  0.6665  0.6577
             baseline_mask  0.6735  0.6684  0.6665  0.6577
             checker        0.6721  0.6665  0.6596  0.6531


### macro_f1  (rows: architecture x variant | cols: noise level)


noise_level                    0.0     0.1     0.2     0.3
architecture variant                                      
gru          baseline       0.4788  0.4768  0.4745  0.4732
             baseline_mask  0.4788  0.4768  0.4745  0.4740
             checker        0.4792  0.4814  0.4775  0.4707
lstm         baseline       0.4887  0.4913  0.4761  0.4716
             baseline_mask  0.4887  0.4913  0.4761  0.4716
             checker        0.4876  0.4914  0.4717  0.4677


### forbidden_mass  (rows: architecture x variant | cols: noise level)


noise_level                    0.0     0.1     0.2     0.3
architecture variant                                      
gru          baseline       0.0033  0.0318  0.0601  0.0874
             baseline_mask  0.0000  0.0000  0.0000  0.0000
             checker        0.0025  0.0224  0.0427  0.0616
lstm         baseline       0.0042  0.0321  0.0592  0.0911
             baseline_mask  0.0000  0.0000  0.0000  0.0000
             checker        0.0031  0.0223  0.0412  0.0639


### violation_rate  (rows: architecture x variant | cols: noise level)


noise_level                 0.0  0.1  0.2     0.3
architecture variant                             
gru          baseline       0.0  0.0  0.0  0.0005
             baseline_mask  0.0  0.0  0.0  0.0000
             checker        0.0  0.0  0.0  0.0005
lstm         baseline       0.0  0.0  0.0  0.0000
             baseline_mask  0.0  0.0  0.0  0.0000
             checker        0.0  0.0  0.0  0.0000

## 8. Conclusions

Read the numbers above for this dataset:

- **Accuracy vs noise** — how far each variant's test accuracy degrades as more
  training labels are corrupted.
- **Conformance vs noise** — `forbidden_mass`/`violation_rate` show whether the
  clean symbolic knowledge keeps predictions on legal transitions even when the
  supervision is dirty. `baseline+mask` enforces this at inference; `checker`
  pushes it into the weights during training.

The side-by-side comparison **across the three datasets** (and the figures) is
produced in the dedicated analysis notebook, which loads
`runs/benchmark_corruption.csv`.